# E1 Stage 2 Stage 0 -- board `B4`, session `s1` -- policy A arm-on setup, no route

Code `/Users/terrancehamilton/reachy-1-2-sim-stage2` = `e6c30b74e09c38e616dee0819a7e205d12c5853e`. One recorded action: `turn_on("r_arm")` + `require_compliance`, gated exactly like a leg; not asserted motion-free. Armon leg name: `stage0-B4-s1-armon`.

In [1]:
# Cell 1 -- connect with literals; hygiene shown; tree identity pinned at generation time
import os, subprocess, sys, time, json, pathlib, traceback
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage2/src"); sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage2/scripts")
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage2/scripts/e1_stage1")
CTRL = pathlib.Path("/Users/terrancehamilton/e1-stage2-B4-s1-2026-09-18/control"); RECORD_ROOT = "/Users/terrancehamilton/e1-stage2-B4-s1-2026-09-18/e1_server_runs"
SCENE = "/Users/terrancehamilton/reachy-1-2-sim-stage2/scenes/e1_boards/B4_pool_box_1_r2c3.yaml"
LEAD_IN_S = 3.0; COMPLIANCE_TIMEOUT_S = 3.0; CYCLE = "stage0-B4-s1"
REPO = "/Users/terrancehamilton/reachy-1-2-sim-stage2"; GENERATED_AT_SHA = "e6c30b74e09c38e616dee0819a7e205d12c5853e"
REQUIRED_SHA = "3d7fc744f81eae14c39b70f22fab3c95fd02626d"; MERGE_TIME_ISO = "2026-09-16T02:20:18Z"
_current_sha = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, timeout=5).stdout.strip()
assert _current_sha == GENERATED_AT_SHA, f"kernel tree {_current_sha} != generated-at tree {GENERATED_AT_SHA} -- regenerate the notebook"
print("REACHY env in this kernel:", {k: v for k, v in os.environ.items() if k.upper().startswith("REACHY")})
assert "REACHY_IP" not in os.environ and "REACHY_ENABLE_MOTION" not in os.environ, "shell hygiene violated"
from reachy_sdk import ReachySDK
HOST, PORT = "localhost", 50051
reachy = ReachySDK(host=HOST, sdk_port=PORT)
print(f"ReachySDK(host={HOST!r}, sdk_port={PORT}) connected at wall {time.time_ns()} mono {time.monotonic_ns()}")
print("python:", sys.executable)


REACHY env in this kernel: {}


ReachySDK(host='localhost', sdk_port=50051) connected at wall 1789792855220235000 mono 367005041603916
python: /Users/terrancehamilton/e1venv/bin/python


In [2]:
# Cell 2 -- motion-client binding check + W4 fresh-server provenance
import e1_identity, provenance, gating, plan
from reachy_ai.motion import rig_routes as R
from reachy_ai.motion import primitives
from reachy_ai.tasks import rig_motion
def _pose(): return {name: float(getattr(reachy.r_arm, name).present_position) for name in R.R_JOINTS}
ident = e1_identity.verify_simulator_identity(host=HOST, port=PORT, scene_path=SCENE, record_root=RECORD_ROOT, read_sdk_joints=_pose)
d = ident.as_dict()
manifest = {}
if ident.run_dir:
    manifest_path = pathlib.Path(ident.run_dir) / "manifest.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
def _git_is_ancestor(a, b):
    return subprocess.run(["git", "-C", REPO, "merge-base", "--is-ancestor", a, b]).returncode == 0
prov_ok, prov_reasons = provenance.check_binding_provenance(manifest, git_is_ancestor=_git_is_ancestor, required_sha=REQUIRED_SHA, merge_time_iso=MERGE_TIME_ISO, generated_at_sha=GENERATED_AT_SHA)
d["provenance_ok"] = prov_ok; d["provenance_reasons"] = prov_reasons
print(json.dumps(d, indent=2, default=str))
BINDING_OK = bool(ident.ok) and prov_ok
(CTRL / (f"binding_ok_{CYCLE}" if BINDING_OK else f"binding_FAIL_{CYCLE}")).write_text(json.dumps(d, default=str))
print("BINDING_OK =", BINDING_OK); print("present:", {k: round(v, 1) for k, v in _pose().items()})
def wait_for(pred, timeout_s, period=0.25):
    return gating.wait_for(CTRL, pred, timeout_s, period)
def start_check(kind):
    p = _pose(); here = R.posture_of(p)
    if kind == "PLACE_ROUTE_start": ok, why = rig_motion.check_start(reachy.r_arm, R.PLACE_ROUTE)
    elif kind == "PRESENT": ok = R.at_pose(p, R.PRESENT, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at PRESENT (gross, 12 deg)"
    elif kind == "REST": ok = R.at_pose(p, R.REST, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at REST (gross, 12 deg)"
    return {"kind": kind, "ok": bool(ok), "why": why, "posture_of": here, "pose": {k: round(v, 1) for k, v in p.items()}}
PREV_OK = BINDING_OK


{
  "ok": true,
  "reasons": [],
  "run_dir": "/Users/terrancehamilton/e1-stage2-B4-s1-2026-09-18/e1_server_runs/run_20260919_043924",
  "manifest": {
    "format_version": 1,
    "started_at": "2026-09-19T04:39:24.641690+00:00",
    "code_sha": "e6c30b74e09c38e616dee0819a7e205d12c5853e",
    "code_sha_dirty": false,
    "model_path": "/Users/terrancehamilton/reachy-1-2-sim-stage2/native_mujoco/model/reachy_1_2.xml",
    "model_sha256": "618ef2499f6207d5e9e72f8b9a4e538b0521008afc1a80f3ecfbab22ab35abb4",
    "scene_path": "/Users/terrancehamilton/reachy-1-2-sim-stage2/scenes/e1_boards/B4_pool_box_1_r2c3.yaml",
    "scene_sha256": "818d8e5755b4a8e892bf4b48a73a0321df717c3dcaf3c2313f9ed367056ab083",
    "scene_revision": "initial",
    "mujoco_version": "3.11.0",
    "python_version": "3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]",
    "platform": "macOS-26.6.2-arm64-arm-64bit-Mach-O",
    "protocol_version": 1,
    "calibration_provenance": "measu

In [3]:
# Stage 0 -- explicit recorded setup action (policy A): arm stiff before reset #1
LEG = {"leg": "stage0-B4-s1-armon", "route": None, "tool": "turn_on", "cycle": CYCLE}
go = wait_for(lambda: (CTRL / "go_stage0-B4-s1-armon").exists(), 1800) if PREV_OK else "not_eligible"
LEG["go"] = go; print("go:", go)
rec = wait_for(lambda: (CTRL / "recorder_stage0-B4-s1-armon.log").exists() and "fly the route now" in (CTRL / "recorder_stage0-B4-s1-armon.log").read_text(), 900) if go == "ready" else go
LEG["recorder_status"] = rec; print("recorder status:", rec)
if rec == "ready":
    time.sleep(LEAD_IN_S)
    LEG["t_start_mono_ns"] = time.monotonic_ns(); LEG["t_start_wall_ns"] = time.time_ns()
    baseline = e1_identity._read_last_state(pathlib.Path(ident.run_dir) / "states.jsonl")
    baseline_cmd_seq = (baseline or {}).get("cmd_seq")
    if isinstance(baseline_cmd_seq, bool) or not isinstance(baseline_cmd_seq, int):
        LEG["outcome"] = "STOP no_valid_baseline_cmd_seq"; LEG["baseline"] = repr(baseline_cmd_seq)
        (CTRL / "stop").write_text(json.dumps(LEG, default=str))
    else:
        reachy.turn_on("r_arm")
        chk = e1_identity.require_compliance(ident.run_dir, R.R_JOINTS, compliant=False,
                                             timeout_s=COMPLIANCE_TIMEOUT_S, min_cmd_seq=baseline_cmd_seq)
        LEG["compliance_check"] = chk.as_dict(); print("compliance:", chk.as_dict())
        LEG["outcome"] = "returned" if chk.ok else "STOP compliance_check"
        if not chk.ok:
            (CTRL / "stop").write_text(json.dumps(chk.as_dict()))
    LEG["t_end_mono_ns"] = time.monotonic_ns(); LEG["t_end_wall_ns"] = time.time_ns()
    LEG["elapsed_s"] = (LEG["t_end_mono_ns"] - LEG["t_start_mono_ns"]) / 1e9
    LEG["end_pose"] = {k: round(v, 1) for k, v in _pose().items()}
    print("outcome:", LEG["outcome"], "elapsed %.1f s" % LEG["elapsed_s"]); print("end pose:", LEG["end_pose"])
else:
    LEG["outcome"] = "not_attempted"
PREV_OK = LEG["outcome"] == "returned"
(CTRL / "stage0-B4-s1-armon_done").write_text(json.dumps(LEG, default=str)); print(json.dumps(LEG, default=str))


go: ready


recorder status: ready


compliance: {'ok': True, 'reasons': [], 'per_joint': {'r_shoulder_pitch': {'compliant': False, 'effort': -6.01161532221978e-18, 'seq': 6276, 'sim_step': 62760}, 'r_shoulder_roll': {'compliant': False, 'effort': -8.671084691228882e-07, 'seq': 6276, 'sim_step': 62760}, 'r_arm_yaw': {'compliant': False, 'effort': 4.2940042705253675e-21, 'seq': 6276, 'sim_step': 62760}, 'r_elbow_pitch': {'compliant': False, 'effort': 1.0764467101525795e-17, 'seq': 6276, 'sim_step': 62760}, 'r_forearm_yaw': {'compliant': False, 'effort': 4.730085250686994e-21, 'seq': 6276, 'sim_step': 62760}, 'r_wrist_pitch': {'compliant': False, 'effort': 1.2900613177608898e-18, 'seq': 6276, 'sim_step': 62760}, 'r_wrist_roll': {'compliant': False, 'effort': -1.9985444794201612e-05, 'seq': 6276, 'sim_step': 62760}, 'r_gripper': {'compliant': False, 'effort': 7.499478157058093e-05, 'seq': 6276, 'sim_step': 62760}}, 'waited_s': 0.052562, 'last_state_age_s': 0.020302458}
outcome: returned elapsed 0.1 s
end pose: {'r_shoulder_p

In [4]:
# Final read-only state; no further motion
print("final pose:", {k: round(v, 1) for k, v in _pose().items()}); print("done at wall", time.time_ns())


final pose: {'r_shoulder_pitch': -0.0, 'r_shoulder_roll': 0.0, 'r_arm_yaw': -0.0, 'r_elbow_pitch': -0.0, 'r_forearm_yaw': -0.0, 'r_wrist_pitch': -0.0, 'r_wrist_roll': 40.1, 'r_gripper': -39.7}
done at wall 1789792890220247000
